In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import sys
import os

sys.path.append(os.path.abspath("../"))

from src.preprocessing import get_preprocessor

df = pd.read_csv("../data/train.csv")
X = df.drop(["SalePrice"], axis=1)
y = df["SalePrice"]

categorical_cols_with_na = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual",
    "GarageCond", "PoolQC", "Fence", "MiscFeature"
]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)
preprocessor = get_preprocessor(categorical_cols_with_na)

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    n_jobs=-1,
    early_stopping_rounds=50,
    random_state=42
)

X_train_prepped = preprocessor.fit_transform(X_train)
X_valid_prepped = preprocessor.transform(X_valid)

model.fit(
    X_train_prepped, y_train,
    eval_set=[(X_valid_prepped, y_valid)],
    verbose=False
)

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

preds = full_pipeline.predict(X_valid)
score = mean_absolute_error(y_valid, preds)
print(f"XGBoost (with Early Stopping) MAE: ${score:,.2f}")

XGBoost (with Early Stopping) MAE: $15,752.55
